In [1]:
import pulp

## SunRay

In [6]:
I = [1, 2, 3] # Silo (Filas)
J = [1, 2, 3, 4] # Molino (Columnas)

c = [
    [10, 2, 20, 11],
    [12, 7, 9, 20],
    [4, 14, 16, 18]
]

d = [5, 15, 15, 15] # Demanda de los 4 Molinos
o = [15, 25, 10]    # Oferta de los 3 Silos

model = pulp.LpProblem("SunRay_Transport", pulp.LpMinimize)

# Variable de decisión: Cantidad enviada del Silo i al Molino j
x = {(i, j): pulp.LpVariable(f"Envio_Silo_{i}_a_Molino_{j}", lowBound=0, cat='Continuous') for i in I for j in J}

# 2. Función Objetivo: Nota el i-1 y j-1 para respetar el índice 0 de Python
model += pulp.lpSum(x[i, j] * c[i-1][j-1] for i in I for j in J), "Costo_Total"

In [7]:
for i in I:
    model += pulp.lpSum(x[i, j] for j in J) <= o[i-1], f"Oferta_Silo_{i}"

# 4. Restricción de Demanda: Lo que llega al Molino 'j' debe ser igual a su demanda
for j in J:
    model += pulp.lpSum(x[i, j] for i in I) == d[j-1], f"Demanda_Molino_{j}"

# Resolver el problema
model.solve()

# Imprimir los resultados
print(f"Estado del modelo: {pulp.LpStatus[model.status]}\n")
print("--- Plan de Envíos Óptimo ---")
for i in I:
    for j in J:
        if x[i, j].varValue > 0:
            print(f"Enviar {x[i, j].varValue} camiones del Silo {i} al Molino {j}")

print(f"\nCosto Total Mínimo: ${pulp.value(model.objective) * 100} dólares")

Estado del modelo: Optimal

--- Plan de Envíos Óptimo ---
Enviar 5.0 camiones del Silo 1 al Molino 2
Enviar 10.0 camiones del Silo 1 al Molino 4
Enviar 10.0 camiones del Silo 2 al Molino 2
Enviar 15.0 camiones del Silo 2 al Molino 3
Enviar 5.0 camiones del Silo 3 al Molino 1
Enviar 5.0 camiones del Silo 3 al Molino 4

Costo Total Mínimo: $43500.0 dólares
